# 03 · CUDA: threads, kernels, memory and a tiled matrix multiply

*Base Model Training in C, C++ and CUDA*, notebook 3 of 4

A graphics processor (GPU) runs thousands of threads at once. Deep learning uses it because its work, such as a matrix product or an activation applied to millions of numbers, splits into many small independent pieces. In this notebook you program an NVIDIA GPU with **CUDA**, C++ with a few extensions. You launch kernels over grids of threads, move data between the computer's memory and the GPU's, write the matrix product twice (once plainly and once with on-chip shared memory), measure them, and write the kernels for a classifier's loss and its bias gradient. The last section runs a whole forward pass of the network on the GPU and checks it against a double-precision computation in Python.

## What you will build

- `blocks_for(n, threads)`: how many blocks cover `n` elements;
- `relu_kernel` and `relu_gpu`: a first kernel, and the host code that allocates, copies, launches, checks and frees;
- `matmul_kernel`: one thread per output element, on a two-dimensional grid;
- `matmul_tiled_kernel`: the same product, staged through shared memory in tiles, with barriers;
- `softmax_xent_kernel` and `column_sums_kernel`: the loss, the logits' gradient and the bias gradient, row by row and column by column.

## Learning objectives

By the end of this notebook you will be able to:

1. explain how a kernel launch maps threads, blocks and grids onto data, and compute a launch configuration with a bounds check;
2. manage device memory and transfers, check every CUDA call and launch for errors, and explain why launches are asynchronous;
3. write elementwise, two-dimensional and per-row kernels, and explain coalesced memory access;
4. use shared memory and `__syncthreads()` to tile a matrix product, and explain why each barrier is needed;
5. time kernels and transfers with CUDA events on a GPU, and interpret the numbers;
6. recognise data races, out-of-bounds writes, unchecked errors, uninitialised memory and non-deterministic atomics.

### How to work through this notebook

Run the cells from top to bottom. At each **✍️ Your turn** you write CUDA C++ inside the string that a small Python function returns: delete the line `raise NotImplementedError`, type your code where the comment says, and run the check cell beneath it. The check compiles your code into a small test program, runs it in a separate process and compares what it prints with what the reference solution prints:

- ✅ means your code matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not compile or does not match yet. The check shows the compiler's message or the first difference. In both cases the notebook carries on with a hidden reference version, so *Run All* always works and you can come back later.

The checks compare numbers with a small tolerance, because the GPU computes in single precision (`float`) and may round sums in a different order. Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 0 for Python:** the standard library only, as in notebooks 1 and 2.
- **An NVIDIA GPU with the CUDA toolkit** runs the kernels for real: `nvcc` compiles them and the GPU executes them. **Google Colab** provides one: choose *Runtime → Change runtime type → T4 GPU* (or any GPU) before running the setup cell. A computer of your own with an NVIDIA GPU, its driver and the CUDA toolkit works too.
- **Without a GPU**, the notebook still runs everything: a **CPU emulator** compiles the same CUDA source with the C++ compiler and runs every thread of every block on the processor, with CUDA's indexing rules, so every check means the same thing. That is how it runs on the course notebook server, which has no GPU. Timings measured with the emulator are processor timings, and the cells say so.
- **Not in the browser preview:** JupyterLite cannot run a compiler.
- You should have finished notebooks 1 and 2, or know a dense layer's forward and backward pass and basic C++.

## Setup

Run these two cells first. The first defines the tools from the earlier notebooks: `build`, `load`, `run`, `mark`, `practice_native` and `practice`. The second chooses the **CUDA backend**:

- with an NVIDIA GPU and `nvcc` present, kernels compile with `nvcc` and run on the GPU;
- otherwise they compile with the C++ compiler and the header `furina_cuda_cpu.h`, the CPU emulator, which is printed in full inside the cell. Before compiling, the notebook rewrites each launch `kernel<<<grid, block>>>(args)` into a call that runs the kernel once for every thread. A kernel that calls `__syncthreads()` runs each block with one operating-system thread per CUDA thread, so its barriers really wait; that is why this notebook keeps such blocks small, at 8 × 8 threads.

The emulator checks results, not speed: it cannot show how fast a kernel is, and it cannot reveal every race that a real GPU might. To force a backend, set the environment variable `FURINA_CUDA_BACKEND` to `gpu` or `emulator` before starting the kernel.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import atexit
import ctypes
import hashlib
import math
import os
import random
import shutil
import signal
import subprocess
import sys
import tempfile
from array import array

if sys.platform == "emscripten":
    raise RuntimeError(
        "This notebook compiles C, C++ and CUDA, which a web browser cannot do. Open it on your "
        "course notebook server, in Google Colab, or in Jupyter on a computer with a C compiler."
    )

CC = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
CXX = shutil.which("c++") or shutil.which("g++") or shutil.which("clang++")
if not (CC and CXX):
    raise RuntimeError("No C and C++ compiler found: install gcc and g++ (or clang), then restart the kernel.")
BUILD_DIR = tempfile.mkdtemp(prefix="furina-native-")
atexit.register(shutil.rmtree, BUILD_DIR, True)
COMPILED = {}  # source hash -> path of the program or library built from it


class CompileError(Exception):
    """The compiler rejected the code. The message is what the compiler printed."""


class RunError(Exception):
    """A compiled program crashed, never finished, or stopped with an error."""


def _trim(text, lines=14):
    kept = text.strip().splitlines()
    return "\n".join(kept[:lines] + (["   ..."] if len(kept) > lines else []))


def mark(source, label):
    """Splice in source so that compiler messages name `label` and count from its first line."""
    return f'\n#line 1 "{label}"\n{source}\n'


def build(source, language="c", shared=False, flags=()):
    """Compile C ("c"), C++ ("c++") or CUDA ("cuda") into a program, or a shared library."""
    key = hashlib.sha256(f"{language}|{shared}|{flags}|{source}".encode()).hexdigest()[:24]
    if key in COMPILED:
        return COMPILED[key]
    output = os.path.join(BUILD_DIR, key + (".so" if shared else ".exe"))
    if language == "cuda":
        command = cuda_command(source, key, output)  # defined in the CUDA notebooks
    else:
        path = os.path.join(BUILD_DIR, key + (".c" if language == "c" else ".cpp"))
        with open(path, "w", encoding="utf-8") as handle:
            handle.write(source)
        if language == "c":
            command = [CC, "-std=c11", "-O2", "-ffp-contract=off", "-Wall", path, "-o", output, "-lm"]
        else:
            command = [CXX, "-std=c++17", "-O2", "-ffp-contract=off", "-Wall", path, "-o", output]
    if shared:
        command[1:1] = ["-shared"] + (["-Xcompiler", "-fPIC"] if command[0] == globals().get("NVCC") else ["-fPIC"])
    command += list(flags)  # last, so that a flag such as -O1 overrides the default
    result = subprocess.run(command, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        raise CompileError(_trim(result.stderr or result.stdout))
    COMPILED[key] = output
    return output


def load(source, language="c"):
    """Compile source into a shared library and load it into this Python process."""
    return ctypes.CDLL(build(source, language, shared=True))


def run(program, timeout=20, env=None):
    """Run a compiled program in its own process and return what it printed."""
    try:
        result = subprocess.run([program], capture_output=True, text=True, timeout=timeout,
                                env=None if env is None else {**os.environ, **env})
    except subprocess.TimeoutExpired:
        raise RunError(f"did not finish within {timeout} seconds: is there a loop that never ends?") from None
    if result.returncode < 0:
        try:
            name = signal.Signals(-result.returncode).name
        except ValueError:
            name = f"signal {-result.returncode}"
        hint = {
            "SIGSEGV": "a read or write outside memory the program owns, such as an index past the end of an array",
            "SIGBUS": "a read or write outside memory the program owns",
            "SIGFPE": "an integer division by zero",
            "SIGABRT": "the program aborted itself, often after finding corrupted memory",
            "SIGTRAP": "the compiler turned undefined behaviour, such as using a null pointer, into a trap",
            "SIGILL": "the compiler turned undefined behaviour, such as using a null pointer, into a trap",
        }.get(name, "")
        detail = f"\n{_trim(result.stderr)}" if result.stderr.strip() else ""
        raise RunError(f"crashed with {name}" + (f" ({hint})" if hint else "") + detail)
    if result.returncode != 0:
        raise RunError(f"stopped with exit status {result.returncode}:\n{_trim(result.stderr + result.stdout)}")
    return result.stdout


def _value(token):
    try:
        return float(token)
    except ValueError:
        return None


def _same(a, b, atol, rtol):
    if a == b:
        return True
    x, y = _value(a), _value(b)
    if x is None or y is None:
        return False
    if math.isnan(x) or math.isnan(y):
        return math.isnan(x) and math.isnan(y)
    return math.isclose(x, y, rel_tol=rtol, abs_tol=atol)


def _short(text, limit=110):
    text = " ".join(text.split())
    return text if len(text) <= limit else text[: limit - 3] + "..."


def compare_output(got, want, atol=1e-9, rtol=1e-9):
    """None when two printouts match line by line, otherwise the first difference."""
    got_lines, want_lines = got.strip().splitlines(), want.strip().splitlines()
    for got_line, want_line in zip(got_lines, want_lines):
        got_label, _, got_values = got_line.partition(":")
        want_label, _, want_values = want_line.partition(":")
        g, w = got_values.split(), want_values.split()
        if got_label != want_label or len(g) != len(w) or not all(_same(a, b, atol, rtol) for a, b in zip(g, w)):
            return (f"for {want_label.strip() or 'a case'}\n   your code printed {_short(got_values) or '(nothing)'}"
                    f"\n   expected          {_short(want_values)}")
    if len(got_lines) < len(want_lines):
        return f"your program stopped before printing {want_lines[len(got_lines)].partition(':')[0]!r}"
    if len(got_lines) > len(want_lines):
        return f"your program printed {len(got_lines) - len(want_lines)} line(s) more than expected"
    return None


REFERENCE_OUTPUT = {}


def practice_native(name, yours, reference, program, language="c", atol=1e-9, rtol=1e-9, timeout=20):
    """Check native code you wrote against the reference.

    `yours` and `reference` return source code. `program(source)` wraps a
    source in a complete program whose main() runs the cases and prints the
    results. Both programs are compiled and run in their own processes, so a
    crash or an endless loop fails the check without stopping this notebook.
    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference, so you can always keep going.
    """
    fallback = "   The rest of the notebook uses the reference version for now."
    key = hashlib.sha256(f"{language}|{program(reference())}".encode()).hexdigest()
    if key not in REFERENCE_OUTPUT:
        REFERENCE_OUTPUT[key] = run(build(program(reference()), language), timeout)
    try:
        source = yours()
        if not isinstance(source, str):
            raise TypeError(f"{name} must return its source code as a string, not {type(source).__name__}")
        got = run(build(program(source), language), timeout)
    except NotImplementedError:
        print(f"✍️ {name}: not written yet.")
        print(fallback)
        return reference
    except CompileError as error:
        print(f"❌ {name}: the compiler rejected your code:\n{error}")
        print(fallback)
        return reference
    except RunError as error:
        print(f"❌ {name}: the check program {error}")
        print(fallback)
        return reference
    except Exception as error:  # a mistake in the Python around the source
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(fallback)
        return reference
    difference = compare_output(got, REFERENCE_OUTPUT[key], atol, rtol)
    if difference:
        print(f"❌ {name}: {difference}")
        print(fallback)
        return reference
    print(f"✅ {name}: matches the reference. From here on, the notebook uses your version.")
    return yours


def _close(got, want, atol):
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want))
    if isinstance(want, float) or isinstance(got, float):
        try:
            return math.isclose(float(got), float(want), rel_tol=1e-9, abs_tol=atol)
        except (TypeError, ValueError):
            return False
    return got == want


def practice(name, yours, reference, cases, atol=1e-9):
    """Check a Python function you wrote against the reference on a few inputs."""
    fallback = "   The rest of the notebook uses the reference version for now."
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(repr(args))}")
                print(f"   you returned {_short(repr(got))}")
                print(f"   expected     {_short(repr(want))}")
                print(fallback)
                return reference
    except NotImplementedError:
        print(f"✍️ {name}: not written yet.")
        print(fallback)
        return reference
    except Exception as error:
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(fallback)
        return reference
    print(f"✅ {name}: matches the reference. From here on, the notebook uses your version.")
    return yours


def address(buffer):
    """The memory address of an array.array, to pass to C as a pointer."""
    return buffer.buffer_info()[0]


def c_list(values, kind="double"):
    """Python numbers as a C initialiser list, such as {1.5, -2.0}."""
    if kind == "int":
        return "{" + ", ".join(str(int(v)) for v in values) + "}"
    suffix = "f" if kind == "float" else ""
    return "{" + ", ".join(f"{float(v)!r}{suffix}" for v in values) + "}"


print(f"C compiler: {CC}\nC++ compiler: {CXX}")

In [ ]:
# ── The CUDA backend: a real GPU when there is one, the CPU emulator otherwise ──
import re

CUDA_CPU_HEADER = r'''
// furina_cuda_cpu.h: runs CUDA C++ on the CPU when no NVIDIA GPU is present.
//
// The course compiles every kernel with nvcc and runs it on the GPU whenever
// the machine has one. This header is the fallback for CPU-only machines: the
// same source then compiles with an ordinary C++ compiler, and every kernel
// runs on the CPU with CUDA's indexing rules, so results can be checked.
//
//   * __global__, __device__ and __host__ functions become ordinary functions.
//   * Before compiling, the notebook rewrites `kernel<<<grid, block>>>(args)`
//     into `furina_launch<SYNC>(kernel, grid, block)(args)`. The launch runs the
//     kernel once for every thread of every block, with threadIdx, blockIdx,
//     blockDim and gridDim set as on a GPU. Blocks run one after another.
//   * A kernel that calls __syncthreads() (SYNC is true) runs each block with
//     one operating-system thread per CUDA thread, so the barrier really
//     waits. Other kernels run their threads one at a time, which is faster.
//   * __shared__ variables become static: blocks run one at a time, so all
//     threads of the running block share one copy, as on a GPU.
//   * cudaMalloc returns host memory filled with 0xFF bytes, which reads as
//     NaN in float or double, so forgetting to initialise memory shows up.
//
// It does not model speed, run blocks in parallel, emulate warps (__shfl_*),
// dynamic shared memory or streams, or find data races. Timings measured
// with it are CPU timings.
#pragma once

#include <cmath>
#include <cstddef>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <pthread.h>
#include <time.h>

#define __global__
#define __device__
#define __host__
#define __forceinline__ inline
#define __launch_bounds__(...)
#define __restrict__ __restrict
#define __shared__ static

struct uint3 { unsigned int x, y, z; };
struct dim3 {
    unsigned int x, y, z;
    dim3(unsigned int x_ = 1, unsigned int y_ = 1, unsigned int z_ = 1) : x(x_), y(y_), z(z_) {}
};

static thread_local uint3 threadIdx = {0, 0, 0};
static thread_local uint3 blockIdx = {0, 0, 0};
static dim3 blockDim, gridDim;
static const int warpSize = 32;

// ------------------------------------------------------------------ errors

enum cudaError_t {
    cudaSuccess = 0,
    cudaErrorInvalidValue = 1,
    cudaErrorMemoryAllocation = 2,
    cudaErrorInvalidConfiguration = 9,
    cudaErrorInvalidMemcpyDirection = 21,
    cudaErrorLaunchOutOfResources = 701,
    cudaErrorLaunchFailure = 719,
};

static cudaError_t furina_error = cudaSuccess;

static inline cudaError_t furina_fail(cudaError_t error) {
    if (furina_error == cudaSuccess) furina_error = error;
    return error;
}

static inline cudaError_t cudaGetLastError() {
    cudaError_t error = furina_error;
    furina_error = cudaSuccess;
    return error;
}

static inline cudaError_t cudaPeekAtLastError() { return furina_error; }

static inline const char* cudaGetErrorString(cudaError_t error) {
    switch (error) {
        case cudaSuccess: return "no error";
        case cudaErrorInvalidValue: return "invalid argument";
        case cudaErrorMemoryAllocation: return "out of memory";
        case cudaErrorInvalidConfiguration: return "invalid configuration argument";
        case cudaErrorInvalidMemcpyDirection: return "invalid copy direction for memcpy";
        case cudaErrorLaunchOutOfResources: return "too many resources requested for launch";
        case cudaErrorLaunchFailure: return "unspecified launch failure";
    }
    return "unknown error";
}

static inline cudaError_t cudaDeviceSynchronize() { return furina_error; }
static inline cudaError_t cudaSetDevice(int) { return cudaSuccess; }
static inline cudaError_t cudaGetDeviceCount(int* count) { *count = 1; return cudaSuccess; }

struct cudaDeviceProp {
    char name[256];
    size_t totalGlobalMem, sharedMemPerBlock;
    int warpSize, maxThreadsPerBlock, multiProcessorCount, major, minor;
    int maxThreadsDim[3], maxGridSize[3];
};

static inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* prop, int) {
    std::memset(prop, 0, sizeof *prop);
    std::snprintf(prop->name, sizeof prop->name, "CPU emulator (no NVIDIA GPU)");
    prop->totalGlobalMem = (size_t)1 << 30;
    prop->sharedMemPerBlock = 48 * 1024;
    prop->warpSize = 32;
    prop->maxThreadsPerBlock = 1024;
    prop->multiProcessorCount = 1;
    prop->maxThreadsDim[0] = 1024; prop->maxThreadsDim[1] = 1024; prop->maxThreadsDim[2] = 64;
    prop->maxGridSize[0] = 2147483647; prop->maxGridSize[1] = 65535; prop->maxGridSize[2] = 65535;
    return cudaSuccess;
}

// ------------------------------------------------------------------ memory

enum cudaMemcpyKind {
    cudaMemcpyHostToHost = 0,
    cudaMemcpyHostToDevice = 1,
    cudaMemcpyDeviceToHost = 2,
    cudaMemcpyDeviceToDevice = 3,
    cudaMemcpyDefault = 4,
};

// Live "device" allocations, so that cudaMemcpy can reject a copy whose
// direction does not match its pointers, as a GPU would.
struct FurinaAllocation { const char* begin; size_t bytes; };
static FurinaAllocation furina_allocations[4096];
static int furina_allocation_count = 0;

static inline bool furina_is_device(const void* pointer) {
    const char* p = static_cast<const char*>(pointer);
    for (int i = 0; i < furina_allocation_count; ++i) {
        const FurinaAllocation& a = furina_allocations[i];
        if (p >= a.begin && p < a.begin + (a.bytes ? a.bytes : 1)) return true;
    }
    return false;
}

static inline cudaError_t cudaMalloc(void** pointer, size_t bytes) {
    if (furina_allocation_count == 4096) return furina_fail(cudaErrorMemoryAllocation);
    void* memory = std::malloc(bytes ? bytes : 1);
    if (!memory) return furina_fail(cudaErrorMemoryAllocation);
    std::memset(memory, 0xFF, bytes);
    furina_allocations[furina_allocation_count++] = {static_cast<const char*>(memory), bytes};
    *pointer = memory;
    return cudaSuccess;
}

template <class T>
static inline cudaError_t cudaMalloc(T** pointer, size_t bytes) {
    return cudaMalloc(reinterpret_cast<void**>(pointer), bytes);
}

static inline cudaError_t cudaFree(void* pointer) {
    if (!pointer) return cudaSuccess;
    for (int i = 0; i < furina_allocation_count; ++i) {
        if (furina_allocations[i].begin == pointer) {
            furina_allocations[i] = furina_allocations[--furina_allocation_count];
            std::free(pointer);
            return cudaSuccess;
        }
    }
    return furina_fail(cudaErrorInvalidValue);
}

static inline cudaError_t cudaMemcpy(void* dst, const void* src, size_t bytes, cudaMemcpyKind kind) {
    bool dst_device = furina_is_device(dst), src_device = furina_is_device(src);
    bool ok = kind == cudaMemcpyDefault
        || (kind == cudaMemcpyHostToDevice && dst_device && !src_device)
        || (kind == cudaMemcpyDeviceToHost && !dst_device && src_device)
        || (kind == cudaMemcpyDeviceToDevice && dst_device && src_device)
        || (kind == cudaMemcpyHostToHost && !dst_device && !src_device);
    if (!ok) return furina_fail(cudaErrorInvalidMemcpyDirection);
    if (bytes) std::memmove(dst, src, bytes);
    return cudaSuccess;
}

static inline cudaError_t cudaMemset(void* pointer, int value, size_t bytes) {
    if (!furina_is_device(pointer)) return furina_fail(cudaErrorInvalidValue);
    std::memset(pointer, value, bytes);
    return cudaSuccess;
}

// ------------------------------------------------------------------ timing

struct FurinaEvent { double milliseconds; };
typedef FurinaEvent* cudaEvent_t;

static inline double furina_now_ms() {
    struct timespec now;
    clock_gettime(CLOCK_MONOTONIC, &now);
    return now.tv_sec * 1e3 + now.tv_nsec / 1e6;
}

static inline cudaError_t cudaEventCreate(cudaEvent_t* event) { *event = new FurinaEvent{0.0}; return cudaSuccess; }
static inline cudaError_t cudaEventRecord(cudaEvent_t event, int stream = 0) { (void)stream; event->milliseconds = furina_now_ms(); return cudaSuccess; }
static inline cudaError_t cudaEventSynchronize(cudaEvent_t) { return cudaSuccess; }
static inline cudaError_t cudaEventElapsedTime(float* ms, cudaEvent_t start, cudaEvent_t end) {
    *ms = static_cast<float>(end->milliseconds - start->milliseconds);
    return cudaSuccess;
}
static inline cudaError_t cudaEventDestroy(cudaEvent_t event) { delete event; return cudaSuccess; }

// ------------------------------------------------- device-side functions

static inline float atomicAdd(float* address, float value) {
    uint32_t* word = reinterpret_cast<uint32_t*>(address);
    uint32_t seen = __atomic_load_n(word, __ATOMIC_RELAXED), wanted;
    float before, after;
    do {
        std::memcpy(&before, &seen, sizeof before);
        after = before + value;
        std::memcpy(&wanted, &after, sizeof wanted);
    } while (!__atomic_compare_exchange_n(word, &seen, wanted, false, __ATOMIC_SEQ_CST, __ATOMIC_RELAXED));
    return before;
}

static inline double atomicAdd(double* address, double value) {
    uint64_t* word = reinterpret_cast<uint64_t*>(address);
    uint64_t seen = __atomic_load_n(word, __ATOMIC_RELAXED), wanted;
    double before, after;
    do {
        std::memcpy(&before, &seen, sizeof before);
        after = before + value;
        std::memcpy(&wanted, &after, sizeof wanted);
    } while (!__atomic_compare_exchange_n(word, &seen, wanted, false, __ATOMIC_SEQ_CST, __ATOMIC_RELAXED));
    return before;
}

static inline int atomicAdd(int* address, int value) { return __atomic_fetch_add(address, value, __ATOMIC_SEQ_CST); }

static inline int min(int a, int b) { return a < b ? a : b; }
static inline int max(int a, int b) { return a > b ? a : b; }
static inline float min(float a, float b) { return a < b ? a : b; }
static inline float max(float a, float b) { return a > b ? a : b; }
static inline double min(double a, double b) { return a < b ? a : b; }
static inline double max(double a, double b) { return a > b ? a : b; }
inline float __expf(float x) { return expf(x); }
inline float __logf(float x) { return logf(x); }
inline float rsqrtf(float x) { return 1.0f / sqrtf(x); }

// ---------------------------------------------------------------- barriers

struct FurinaBlock {
    pthread_mutex_t mutex;
    pthread_cond_t changed;
    unsigned int live;          // threads of this block that have not returned
    unsigned int arrived;       // threads waiting at the current barrier
    unsigned long generation;   // how many times the barrier has opened
};

static FurinaBlock* furina_block = nullptr;  // the block running on threads, if any
struct FurinaUnexpectedSync {};

static inline void furina_open(FurinaBlock* block) {
    block->arrived = 0;
    block->generation++;
    pthread_cond_broadcast(&block->changed);
}

static inline void __syncthreads() {
    FurinaBlock* block = furina_block;
    if (!block) throw FurinaUnexpectedSync{};
    pthread_mutex_lock(&block->mutex);
    unsigned long generation = block->generation;
    if (++block->arrived >= block->live) {
        furina_open(block);
    } else {
        while (generation == block->generation) pthread_cond_wait(&block->changed, &block->mutex);
    }
    pthread_mutex_unlock(&block->mutex);
}

// A thread that has returned counts as arrived at every later barrier, so a
// block whose threads leave early does not wait forever.
static inline void furina_thread_finished(FurinaBlock* block) {
    pthread_mutex_lock(&block->mutex);
    block->live--;
    if (block->arrived > 0 && block->arrived >= block->live) furina_open(block);
    pthread_mutex_unlock(&block->mutex);
}

template <class Body>
struct FurinaThread {
    Body* body;
    FurinaBlock* block;
    uint3 thread, index;
};

template <class Body>
static void* furina_thread_main(void* raw) {
    FurinaThread<Body>* start = static_cast<FurinaThread<Body>*>(raw);
    threadIdx = start->thread;
    blockIdx = start->index;
    (*start->body)();
    furina_thread_finished(start->block);
    return nullptr;
}

// Runs one block with one OS thread per CUDA thread; false if the threads
// could not all be started (the notebook server limits how many may run).
template <class Body>
static bool furina_run_block_on_threads(Body& body, uint3 index, dim3 block) {
    unsigned int count = block.x * block.y * block.z;
    FurinaBlock state;
    pthread_mutex_init(&state.mutex, nullptr);
    pthread_cond_init(&state.changed, nullptr);
    state.live = count;
    state.arrived = 0;
    state.generation = 0;
    FurinaThread<Body>* starts = new FurinaThread<Body>[count];
    pthread_t* threads = new pthread_t[count];
    pthread_attr_t attributes;
    pthread_attr_init(&attributes);
    pthread_attr_setstacksize(&attributes, 256 * 1024);
    furina_block = &state;
    unsigned int started = 0;
    for (; started < count; ++started) {
        uint3 thread = {started % block.x, (started / block.x) % block.y, started / (block.x * block.y)};
        starts[started] = {&body, &state, thread, index};
        if (pthread_create(&threads[started], &attributes, furina_thread_main<Body>, &starts[started]) != 0) break;
    }
    if (started < count) {
        // Threads already running may be waiting for the ones that never
        // started: count those as finished so that the block can drain.
        pthread_mutex_lock(&state.mutex);
        state.live -= count - started;
        if (state.arrived > 0 && state.arrived >= state.live) furina_open(&state);
        pthread_mutex_unlock(&state.mutex);
        std::fprintf(stderr,
            "CPU emulator: could only start %u of the %u threads of a block that calls __syncthreads(); "
            "use a smaller block, such as 8 x 8.\n", started, count);
    }
    for (unsigned int i = 0; i < started; ++i) pthread_join(threads[i], nullptr);
    furina_block = nullptr;
    pthread_attr_destroy(&attributes);
    pthread_cond_destroy(&state.changed);
    pthread_mutex_destroy(&state.mutex);
    delete[] threads;
    delete[] starts;
    return started == count;
}

// ------------------------------------------------------------------ launch

template <bool Sync, class Body>
static void furina_run(dim3 grid, dim3 block, size_t shared, Body body) {
    unsigned long long threads = 1ULL * block.x * block.y * block.z;
    if (shared != 0) {
        std::fprintf(stderr, "CPU emulator: dynamic shared memory (a third launch argument) is not supported.\n");
        furina_fail(cudaErrorInvalidValue);
        return;
    }
    if (threads == 0 || threads > 1024 || block.z > 64 || grid.x == 0 || grid.y == 0 || grid.z == 0
        || grid.y > 65535 || grid.z > 65535) {
        furina_fail(cudaErrorInvalidConfiguration);
        return;
    }
    gridDim = grid;
    blockDim = block;
    for (unsigned int bz = 0; bz < grid.z; ++bz)
        for (unsigned int by = 0; by < grid.y; ++by)
            for (unsigned int bx = 0; bx < grid.x; ++bx) {
                uint3 index = {bx, by, bz};
                if (Sync) {
                    if (!furina_run_block_on_threads(body, index, block)) {
                        furina_fail(cudaErrorLaunchOutOfResources);
                        return;
                    }
                    continue;
                }
                blockIdx = index;
                try {
                    for (unsigned int tz = 0; tz < block.z; ++tz)
                        for (unsigned int ty = 0; ty < block.y; ++ty)
                            for (unsigned int tx = 0; tx < block.x; ++tx) {
                                threadIdx = {tx, ty, tz};
                                body();
                            }
                } catch (FurinaUnexpectedSync&) {
                    std::fprintf(stderr, "CPU emulator: __syncthreads() was called outside a kernel that "
                                         "contains it; call it directly in the kernel's body.\n");
                    furina_fail(cudaErrorLaunchFailure);
                    return;
                }
            }
}

template <bool Sync, class... Params>
struct FurinaLaunch {
    void (*kernel)(Params...);
    dim3 grid, block;
    size_t shared;
    template <class... Args>
    void operator()(Args... args) const {
        void (*k)(Params...) = kernel;
        furina_run<Sync>(grid, block, shared, [=]() { k(args...); });
    }
};

template <bool Sync, class... Params>
static FurinaLaunch<Sync, Params...> furina_launch(void (*kernel)(Params...), dim3 grid, dim3 block,
                                                   size_t shared = 0, int stream = 0) {
    (void)stream;
    return {kernel, grid, block, shared};
}
'''

_COMMENT = re.compile(r"//[^\n]*|/\*.*?\*/|\"(?:\\.|[^\"\\\n])*\"|'(?:\\.|[^'\\\n])*'", re.S)
_DEFINITION = re.compile(r"__(?:global|device)__[^;{(]*?\b([A-Za-z_]\w*)\s*\(")
_LAUNCH = re.compile(r"\b([A-Za-z_]\w*)\s*<<<(.*?)>>>\s*\(", re.S)


def kernels_that_sync(source):
    # Kernels and __device__ functions that call __syncthreads(), directly or through a helper.
    code = _COMMENT.sub(lambda m: m.group(0) if m.group(0)[0] in "\"'" else " " * len(m.group(0)), source)
    bodies = {}
    for match in _DEFINITION.finditer(code):
        depth, i = 1, match.end()
        while i < len(code) and depth:
            depth += {"(": 1, ")": -1}.get(code[i], 0)
            i += 1
        while i < len(code) and code[i].isspace():
            i += 1
        if i < len(code) and code[i] == "{":
            start, depth = i, 0
            while i < len(code):
                depth += {"{": 1, "}": -1}.get(code[i], 0)
                i += 1
                if depth == 0:
                    break
            bodies[match.group(1)] = code[start:i]
    syncing = {name for name, body in bodies.items() if "__syncthreads" in body}
    grew = True
    while grew:
        grew = False
        for name, body in bodies.items():
            if name not in syncing and any(re.search(rf"\b{helper}\s*\(", body) for helper in syncing):
                syncing.add(name)
                grew = True
    return syncing


def emulate_launches(source):
    # Rewrite kernel<<<grid, block>>>(args) as furina_launch<sync>(kernel, grid, block)(args).
    syncing = kernels_that_sync(source)
    return _LAUNCH.sub(
        lambda m: f"furina_launch<{'true' if m.group(1) in syncing else 'false'}>({m.group(1)}, {m.group(2)})(",
        source)


def _find_gpu():
    nvcc = shutil.which("nvcc") or next(
        (path for path in ("/usr/local/cuda/bin/nvcc",) if os.path.exists(path)), None)
    smi = shutil.which("nvidia-smi")
    if not (nvcc and smi):
        return None, None
    try:
        listing = subprocess.run([smi, "-L"], capture_output=True, text=True, timeout=30).stdout
    except (OSError, subprocess.SubprocessError):
        return None, None
    gpus = [line for line in listing.splitlines() if line.startswith("GPU ")]
    return (nvcc, gpus[0].split(" (UUID")[0]) if gpus else (None, None)


def cuda_command(source, key, output):
    path = os.path.join(BUILD_DIR, key + (".cu" if CUDA_BACKEND == "gpu" else ".cpp"))
    with open(path, "w", encoding="utf-8") as handle:
        handle.write(source if CUDA_BACKEND == "gpu" else emulate_launches(source))
    if CUDA_BACKEND == "gpu":
        return [NVCC, "-O2", f"-arch={NVCC_ARCH}", path, "-o", output]
    return [CXX, "-std=c++17", "-O2", "-ffp-contract=off", "-pthread", "-include", HEADER_PATH, path, "-o", output]


HEADER_PATH = os.path.join(BUILD_DIR, "furina_cuda_cpu.h")
with open(HEADER_PATH, "w", encoding="utf-8") as handle:
    handle.write(CUDA_CPU_HEADER)

_choice = os.environ.get("FURINA_CUDA_BACKEND", "auto").strip().lower()
NVCC, GPU_NAME = (None, None) if _choice == "emulator" else _find_gpu()
if _choice == "gpu" and not NVCC:
    raise RuntimeError("FURINA_CUDA_BACKEND=gpu, but no NVIDIA GPU with nvcc was found.")
CUDA_BACKEND = "gpu" if NVCC else "emulator"
NVCC_ARCH = "native"
if CUDA_BACKEND == "gpu":
    probe = os.path.join(BUILD_DIR, "probe.cu")
    with open(probe, "w", encoding="utf-8") as handle:
        handle.write("__global__ void k() {}\nint main() { k<<<1, 1>>>(); return cudaDeviceSynchronize(); }\n")
    if subprocess.run([NVCC, "-arch=native", probe, "-o", probe + ".exe"], capture_output=True).returncode:
        NVCC_ARCH = "sm_60"  # an nvcc older than 11.6 has no -arch=native
    print(f"CUDA backend: NVIDIA GPU, {GPU_NAME}. Kernels compile with {NVCC} and run on the GPU.")
else:
    print("CUDA backend: CPU emulator (no NVIDIA GPU with nvcc was found). Kernels compile with the C++ "
          "compiler and run on the CPU with CUDA's indexing rules. Results are checked as on a GPU; "
          "timings are CPU timings, not GPU performance.")

Every check program starts with the same prelude: `CUDA_CHECK`, a macro that stops the program with a readable message when a CUDA call fails, `show` and `fill`, as before but for `float`, and `TILE`, the tile width used in section 5.

In [ ]:
CHECK_CU = r"""
#include <cmath>
#include <cstdio>
#include <cstdlib>

#ifndef TILE
#define TILE 8          // 8 x 8 = 64 threads per tiled block; GPUs usually use 16 or 32
#endif

// Stops the check program with a readable message when a CUDA call fails.
#define CUDA_CHECK(call)                                                                   \
    do {                                                                                   \
        cudaError_t error_ = (call);                                                       \
        if (error_ != cudaSuccess) {                                                       \
            std::printf("CUDA error: %s (line %d)\n", cudaGetErrorString(error_), __LINE__); \
            std::exit(1);                                                                  \
        }                                                                                  \
    } while (0)

static void show(const char* label, const float* x, int n)
{
    std::printf("%s:", label);
    for (int i = 0; i < n; ++i)
        std::printf(" %.9g", x[i]);
    std::printf("\n");
}

static void fill(float* x, int n, float seed)         // the same test numbers on every machine
{
    for (int i = 0; i < n; ++i)
        x[i] = sinf(seed + 0.37f * i);
}
"""


def cuda_program(pieces, main):
    """A check program: the prelude, each (label, source) piece, then main()."""
    return CHECK_CU + "".join(mark(source, label) for label, source in pieces) + mark(main, "the check")

## 1 · The execution model: grids, blocks and threads

A CUDA program has two sides. The **host** is the CPU, running ordinary C++. The **device** is the GPU, with its own memory. A **kernel** is a function marked `__global__`: the host launches it, and the device runs it once in every thread of a **grid**.

```cpp
__global__ void add_one(float* x, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // this thread's global index
    if (i < n)                                       // the grid may be larger than the data
        x[i] += 1.0f;
}

add_one<<<blocks, threads_per_block>>>(d_x, n);      // the launch configuration
```

- Threads are grouped into **blocks** of up to 1,024 threads, and blocks into the grid. Inside a kernel, `threadIdx` is the thread's position in its block, `blockIdx` the block's position in the grid, and `blockDim` and `gridDim` their sizes. Each has `.x`, `.y` and `.z` parts, for data with up to three dimensions.
- `blockIdx.x * blockDim.x + threadIdx.x` numbers the threads of a one-dimensional grid 0, 1, 2, … across all blocks, so thread `i` can handle element `i`.
- The number of blocks must cover all `n` elements, so it is rounded **up**, and the last block usually has threads with nothing to do: the test `if (i < n)` keeps them from writing past the end of the array.
- The GPU runs each block on one of its **streaming multiprocessors** (SMs), in groups of 32 threads called **warps** that execute the same instruction together. Blocks run in no particular order and cannot wait for each other; threads *within* a block can share fast memory and synchronise.

The cell below asks the device about itself.

In [ ]:
DEVICE_QUERY = r"""
#include <cstdio>
int main()
{
    cudaDeviceProp prop;
    cudaError_t error = cudaGetDeviceProperties(&prop, 0);
    if (error != cudaSuccess) {
        std::printf("no CUDA device: %s\n", cudaGetErrorString(error));
        return 1;
    }
    std::printf("device:                    %s\n", prop.name);
    std::printf("compute capability:        %d.%d\n", prop.major, prop.minor);
    std::printf("streaming multiprocessors: %d\n", prop.multiProcessorCount);
    std::printf("warp size:                 %d threads\n", prop.warpSize);
    std::printf("threads per block, at most: %d\n", prop.maxThreadsPerBlock);
    std::printf("shared memory per block:   %zu KiB\n", prop.sharedMemPerBlock / 1024);
    std::printf("device memory:             %.1f GiB\n", prop.totalGlobalMem / 1073741824.0);
    return 0;
}
"""
print(run(build(DEVICE_QUERY, "cuda")))

**What to notice.** On a Colab T4 you would see 40 multiprocessors, warps of 32 threads, at most 1,024 threads per block, 48 KiB of shared memory per block and about 15 GiB of device memory; other GPUs differ. With the emulator, the cell prints the emulator's stand-in values. Every GPU runs at most 1,024 threads per block, so a launch asking for more fails, as section 3 shows.

### ✍️ Your turn: `blocks_for`

Write `blocks_for(n, threads)` in Python:

- **inputs:** `n`, the number of elements, and `threads`, the threads per block;
- **output:** the smallest number of blocks whose threads cover all `n` elements, as an `int`.

Hint: integer division rounds down, and $\lceil n / t \rceil$ = `(n + t - 1) // t` for positive `t`. Zero elements need zero blocks.

*Type it rather than pasting it: typing is how the arithmetic sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def blocks_for_reference(n, threads):
    return (n + threads - 1) // threads

In [ ]:
def blocks_for(n, threads):
    """The number of blocks of `threads` threads that cover n elements, rounded up."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
blocks_for = practice("blocks_for", blocks_for, blocks_for_reference, cases=[
    (1000, 256),
    (1024, 256),     # an exact multiple: no extra block
    (1, 256),
    (257, 128),
    (0, 256),        # edge case: nothing to do
])

**Expected behaviour.** 1,000 elements need 4 blocks of 256 threads (1,024 threads, 24 of them idle); 1,024 elements need exactly 4; a single element needs 1 block and 257 elements need 3 blocks of 128. With 0 elements there is nothing to launch. A grid of 0 blocks is not allowed: such a launch fails with *invalid configuration argument*.

## 2 · Memory and the host's side of a launch

The GPU cannot read the host's memory directly in the ordinary case, and the host cannot read the GPU's. Running a kernel on host data takes five steps, the pattern of every CUDA program:

1. **Allocate device memory** with `cudaMalloc(&d_x, bytes)`, which sets the pointer `d_x` to device memory. By convention, device pointers start with `d_`.
2. **Copy the inputs to the device** with `cudaMemcpy(d_x, x, bytes, cudaMemcpyHostToDevice)`.
3. **Launch the kernel** with its configuration, then call `cudaGetLastError()`: a launch returns nothing, and a bad configuration is reported only through that call.
4. **Copy the results back** with `cudaMemcpy(y, d_y, bytes, cudaMemcpyDeviceToHost)`. A launch is **asynchronous**: it returns to the host at once, while the GPU works. The copy back waits for the kernel to finish, so after it the results are complete.
5. **Free** each allocation with `cudaFree`.

Every CUDA runtime call returns a `cudaError_t`, and `cudaGetErrorString` turns it into text. The prelude's `CUDA_CHECK(call)` stops the check program with that text and the line number; use it around every call. Errors that happen *while* a kernel runs, such as an invalid address, are reported by the next call that waits for the GPU, usually the copy back.

Transfers are slow compared with the GPU's own memory: a PCIe link moves roughly 10 to 25 GB/s, while device memory delivers hundreds of GB/s. Real training therefore copies its data and weights to the device once and keeps them there; notebook 4 does exactly that.

### ✍️ Your turn: `relu_kernel` and `relu_gpu`

Complete two functions:

- the kernel `relu_kernel(x, y, n)`: thread `i` sets `y[i]` to `x[i]` when it is positive and to `0.0f` otherwise, if `i < n`;
- the host function `relu_gpu(x, y, n)`, where `x` and `y` are **host** arrays of `n` floats: the five steps above, with 256 threads per block and `CUDA_CHECK` around every CUDA call, including `cudaGetLastError()` after the launch. When `n` is 0, return at once: a grid of zero blocks is invalid.

The check runs your kernel alone on a device array followed by 64 guard values, to catch any write past the end, and then calls `relu_gpu`.

*Type it rather than pasting it: typing is how the pattern sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def relu_gpu_cu_reference():
    return r"""
__global__ void relu_kernel(const float* x, float* y, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        y[i] = x[i] > 0.0f ? x[i] : 0.0f;
}

void relu_gpu(const float* x, float* y, int n)
{
    if (n == 0)
        return;
    size_t bytes = n * sizeof(float);
    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, bytes));
    CUDA_CHECK(cudaMalloc(&d_y, bytes));
    CUDA_CHECK(cudaMemcpy(d_x, x, bytes, cudaMemcpyHostToDevice));
    int threads = 256, blocks = (n + threads - 1) / threads;
    relu_kernel<<<blocks, threads>>>(d_x, d_y, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(y, d_y, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));
}
"""

In [ ]:
def relu_gpu_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete both functions below
    return r"""
__global__ void relu_kernel(const float* x, float* y, int n)
{
    /* ✍️ Your code here: the global index, the bounds check, the assignment */
}

void relu_gpu(const float* x, float* y, int n)
{
    /* ✍️ Your code here: allocate, copy in, launch with 256 threads per block, check, copy out, free */
}
"""

In [ ]:
RELU_MAIN = r"""
int main()
{
    const int n = 1000, guard = 64;
    float x[n + guard], y[n + guard];
    for (int i = 0; i < n + guard; ++i) {
        x[i] = (i % 7) - 3.0f;
        y[i] = -99.0f;                                 // guard values after the end
    }
    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, sizeof x));
    CUDA_CHECK(cudaMalloc(&d_y, sizeof y));
    CUDA_CHECK(cudaMemcpy(d_x, x, sizeof x, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, y, sizeof y, cudaMemcpyHostToDevice));
    relu_kernel<<<(n + 255) / 256, 256>>>(d_x, d_y, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(y, d_y, sizeof y, cudaMemcpyDeviceToHost));
    int outside = 0;
    for (int i = n; i < n + guard; ++i)
        outside += y[i] != -99.0f;
    show("kernel, first 8 of 1000", y, 8);
    show("kernel, last 4 of 1000", y + n - 4, 4);
    std::printf("kernel, values written past the end: %d\n", outside);
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));

    float host_y[n];
    relu_gpu(x, host_y, n);
    float total = 0.0f;
    for (int i = 0; i < n; ++i)
        total += host_y[i];
    show("relu_gpu, sum of 1000", &total, 1);
    float one = -5.0f, out = 99.0f;
    relu_gpu(&one, &out, 1);
    show("relu_gpu, n = 1", &out, 1);
    float untouched = 7.0f;
    relu_gpu(x, &untouched, 0);
    show("relu_gpu, n = 0", &untouched, 1);
    return 0;
}
"""


def relu_program(source):
    return cuda_program([("your relu", source)], RELU_MAIN)


relu_gpu_cu = practice_native("relu_kernel and relu_gpu", relu_gpu_cu, relu_gpu_cu_reference, relu_program,
                              language="cuda", atol=1e-6, rtol=1e-6)

**Expected behaviour.** The kernel writes `0 0 0 0 1 2 3 0` at the start and nothing past the end, and `relu_gpu` returns the same values: their sum is 855. With one negative element the output is 0, and with `n = 0` nothing changes. If the check reports values written past the end, the bounds check is missing; if it stops with `CUDA error: invalid configuration argument`, `relu_gpu` launched a grid of zero blocks.

### 🧪 Try it: what goes wrong silently

Two mistakes produce no error message unless you look for them. The program below makes both on purpose:

- it launches a block of 2,048 threads, more than any GPU allows, and does **not** check the launch at first;
- it launches a kernel **without** a bounds check on 1,000 elements, with 4 blocks of 256 threads, into an array followed by guard values.

Predict what each line will print, then run it.

In [ ]:
SILENT_BUGS = r"""
__global__ void relu_unchecked(const float* x, float* y, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    y[i] = x[i] > 0.0f ? x[i] : 0.0f;                  // the bug: no test of i < n
}

int main()
{
    const int n = 1000, guard = 64;
    float x[n + guard], y[n + guard];
    for (int i = 0; i < n + guard; ++i) {
        x[i] = 1.0f;
        y[i] = -99.0f;
    }
    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, sizeof x));
    CUDA_CHECK(cudaMalloc(&d_y, sizeof y));
    CUDA_CHECK(cudaMemcpy(d_x, x, sizeof x, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, y, sizeof y, cudaMemcpyHostToDevice));

    relu_kernel<<<1, 2048>>>(d_x, d_y, n);              // too many threads per block
    cudaMemcpy(y, d_y, sizeof y, cudaMemcpyDeviceToHost);   // unchecked on purpose
    std::printf("after the 2048-thread launch, y[0] is still %g\n", y[0]);
    std::printf("cudaGetLastError() then reports: %s\n", cudaGetErrorString(cudaGetLastError()));

    relu_unchecked<<<4, 256>>>(d_x, d_y, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(y, d_y, sizeof y, cudaMemcpyDeviceToHost));
    int outside = 0;
    for (int i = n; i < n + guard; ++i)
        outside += y[i] != -99.0f;
    std::printf("without a bounds check, %d values were written past the end of the data\n", outside);
    return 0;
}
"""
print(run(build(cuda_program([("relu", relu_gpu_cu())], SILENT_BUGS), "cuda")))

**What to notice.** The oversized launch never ran: `y[0]` is still −99, and only `cudaGetLastError()` reveals *invalid configuration argument*. Without that check, a program would carry on with stale data. The kernel without a bounds check wrote 24 values past the end of the data, one for each idle thread of the last block. Here they landed on the guard values; in a real program they would overwrite whatever array came next, or crash, depending on the allocation. On a GPU, NVIDIA's `compute-sanitizer` finds such writes the way AddressSanitizer did in notebook 1.

## 3 · Matrix multiplication with one thread per output

For $C = AB$ with $A$ of shape $n \times k$ and $B$ of shape $k \times m$, the natural mapping gives each output element $C_{ij}$ its own thread, on a **two-dimensional** grid of two-dimensional blocks:

```cpp
dim3 block(16, 16);                                   // 256 threads per block
dim3 grid((m + 15) / 16, (n + 15) / 16);              // x covers the columns, y the rows
matmul_kernel<<<grid, block>>>(d_A, d_B, d_C, n, k, m);
```

Inside the kernel, `col = blockIdx.x * blockDim.x + threadIdx.x` and `row = blockIdx.y * blockDim.y + threadIdx.y`, and both must be tested against the matrix's size, because the blocks overhang its edges.

The `x` direction goes along a row on purpose. The 32 threads of a warp have consecutive `threadIdx.x`, so they handle 32 consecutive columns. When they read `B[r * m + col]` and write `C[row * m + col]`, they touch 32 consecutive floats, and the GPU combines those accesses into a few wide memory transactions. This is called **coalescing**, and it is the GPU's version of notebook 1's stride-1 loop. All 32 threads read the same `A[row * k + r]`, which the hardware serves as one read.

### ✍️ Your turn: `matmul_kernel`

Complete `matmul_kernel(A, B, C, n, k, m)`, where all three matrices are row-major arrays in device memory:

- compute `row` and `col` from the block and thread indices as above;
- if `row < n` and `col < m`, accumulate $\sum_r$ `A[row * k + r] * B[r * m + col]` in a local `float` and store it in `C[row * m + col]`.

The check launches it with 16 × 16 blocks, on sizes that do not divide by 16, and with guard values after `C`.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def matmul_cu_reference():
    return r"""
__global__ void matmul_kernel(const float* A, const float* B, float* C, int n, int k, int m)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < m) {
        float acc = 0.0f;
        for (int r = 0; r < k; ++r)
            acc += A[row * k + r] * B[r * m + col];
        C[row * m + col] = acc;
    }
}
"""

In [ ]:
def matmul_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void matmul_kernel(const float* A, const float* B, float* C, int n, int k, int m)
{
    /* ✍️ Your code here: row and col, the bounds test, the dot product */
}
"""

In [ ]:
MULTIPLY_HELPER = r"""
// Multiplies generated n x k and k x m matrices with `kernel` and prints C, plus the
// number of guard values after C that changed.
static void multiply(const char* label, int n, int k, int m, bool tiled)
{
    const int guard = 64;
    float* A = (float*)std::malloc(n * k * sizeof(float));
    float* B = (float*)std::malloc(k * m * sizeof(float));
    float* C = (float*)std::malloc((n * m + guard) * sizeof(float));
    fill(A, n * k, 0.3f);
    fill(B, k * m, 1.7f);
    for (int i = 0; i < n * m + guard; ++i)
        C[i] = -99.0f;
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, n * k * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_B, k * m * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_C, (n * m + guard) * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_A, A, n * k * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, B, k * m * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_C, C, (n * m + guard) * sizeof(float), cudaMemcpyHostToDevice));
    LAUNCH
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(C, d_C, (n * m + guard) * sizeof(float), cudaMemcpyDeviceToHost));
    int outside = 0;
    for (int i = n * m; i < n * m + guard; ++i)
        outside += C[i] != -99.0f;
    show(label, C, n * m);
    std::printf("%s, values written past the end: %d\n", label, outside);
    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaFree(d_C));
    std::free(A);
    std::free(B);
    std::free(C);
}

int main()
{
    multiply("2x3 times 3x2", 2, 3, 2, false);
    multiply("37x29 times 29x23", 37, 29, 23, false);
    multiply("8x8 times 8x8", 8, 8, 8, false);
    multiply("5x17 times 17x3", 5, 17, 3, false);
    multiply("1x1 times 1x1", 1, 1, 1, false);
    return 0;
}
"""
NAIVE_LAUNCH = r"""dim3 block(16, 16), grid((m + 15) / 16, (n + 15) / 16);
    matmul_kernel<<<grid, block>>>(d_A, d_B, d_C, n, k, m);"""


def matmul_program(source):
    return cuda_program([("your matmul_kernel", source)], MULTIPLY_HELPER.replace("LAUNCH", NAIVE_LAUNCH))


matmul_cu = practice_native("matmul_kernel", matmul_cu, matmul_cu_reference, matmul_program, language="cuda",
                            atol=1e-5, rtol=1e-5)

**Expected behaviour.** ✅, with no values written past the end for any size. If only the 8 × 8 case or the square ones pass, the row and column are swapped somewhere: `blockIdx.x` and `threadIdx.x` must go with the columns. If values appear past the end, a bounds test is missing for the threads that overhang the matrix.

## 4 · Shared memory, tiles and barriers

In `matmul_kernel`, every element of $A$ is read from device memory by $m$ different threads and every element of $B$ by $n$ threads. Device memory is far slower than the arithmetic, so a matrix product written this way waits on memory most of the time.

Each multiprocessor has a small, fast **shared memory**, visible to all threads of a block, declared inside a kernel with `__shared__`. The **tiled** product uses it like a cache that the block fills by hand. Each block computes a $T \times T$ tile of $C$ with $T \times T$ threads, one per output element, and walks along the shared dimension $k$ in steps of $T$:

1. every thread loads **one** element of the current $T \times T$ tile of $A$ and one of $B$ into shared memory, writing 0 where the tile hangs over the edge of a matrix;
2. `__syncthreads()`, a **barrier**: no thread continues until every thread of the block has arrived, so the tiles are complete before anyone reads them;
3. every thread adds the $T$ products of its row of the $A$ tile and its column of the $B$ tile to its own sum, reading only shared memory;
4. `__syncthreads()` again, so that no thread overwrites a tile with the next step's data while another thread is still reading it.

Each element now comes from device memory once per tile instead of once per thread, which reduces the reads by a factor of $T$. Two rules make the barriers safe. Every thread of the block must reach every `__syncthreads()`, so the kernel must not `return` early: threads outside the matrix still load zeros and only skip the final store. And the second barrier is not optional, as the experiment below shows.

This notebook uses $T$ = `TILE` = 8, so 64 threads per block, a size the CPU emulator can run as real threads. On a GPU, 16 or 32 is usual; section 5 compiles with 16 when it runs on one.

### ✍️ Your turn: `matmul_tiled_kernel`

Complete `matmul_tiled_kernel(A, B, C, n, k, m)`, launched with blocks of `TILE` × `TILE` threads:

- declare `__shared__ float As[TILE][TILE];` and `Bs[TILE][TILE]`;
- `row = blockIdx.y * TILE + threadIdx.y`, `col = blockIdx.x * TILE + threadIdx.x`, and a sum `acc = 0.0f`;
- for each step `t` from 0 to $\lceil k / T \rceil - 1$: load `A[row * k + t * TILE + threadIdx.x]` into `As[threadIdx.y][threadIdx.x]` (or 0 when the row or that column is outside $A$) and `B[(t * TILE + threadIdx.y) * m + col]` into `Bs[threadIdx.y][threadIdx.x]` (or 0 outside $B$); barrier; add `As[threadIdx.y][r] * Bs[r][threadIdx.x]` for `r` from 0 to `TILE - 1`; barrier;
- after the loop, store `acc` in `C[row * m + col]` if `row < n` and `col < m`.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def matmul_tiled_cu_reference():
    return r"""
__global__ void matmul_tiled_kernel(const float* A, const float* B, float* C, int n, int k, int m)
{
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float acc = 0.0f;
    for (int t = 0; t < (k + TILE - 1) / TILE; ++t) {
        int a_col = t * TILE + threadIdx.x;
        int b_row = t * TILE + threadIdx.y;
        As[threadIdx.y][threadIdx.x] = (row < n && a_col < k) ? A[row * k + a_col] : 0.0f;
        Bs[threadIdx.y][threadIdx.x] = (b_row < k && col < m) ? B[b_row * m + col] : 0.0f;
        __syncthreads();
        for (int r = 0; r < TILE; ++r)
            acc += As[threadIdx.y][r] * Bs[r][threadIdx.x];
        __syncthreads();
    }
    if (row < n && col < m)
        C[row * m + col] = acc;
}
"""

In [ ]:
def matmul_tiled_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void matmul_tiled_kernel(const float* A, const float* B, float* C, int n, int k, int m)
{
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    /* ✍️ Your code here: row, col and acc; the loop over tiles with its two barriers; the final store */
}
"""

In [ ]:
TILED_LAUNCH = r"""dim3 block(TILE, TILE), grid((m + TILE - 1) / TILE, (n + TILE - 1) / TILE);
    matmul_tiled_kernel<<<grid, block>>>(d_A, d_B, d_C, n, k, m);"""


def matmul_tiled_program(source):
    return cuda_program([("your matmul_tiled_kernel", source)], MULTIPLY_HELPER.replace("LAUNCH", TILED_LAUNCH))


matmul_tiled_cu = practice_native("matmul_tiled_kernel", matmul_tiled_cu, matmul_tiled_cu_reference,
                                  matmul_tiled_program, language="cuda", atol=1e-5, rtol=1e-5)

**Expected behaviour.** ✅, with the same products as `matmul_kernel` up to rounding. If your kernel hangs, it probably returns early, before a barrier, for threads outside the matrix; if the edges are wrong but the 8 × 8 case is right, look at the zero padding of the tiles.

### 🧪 Try it: remove the second barrier

The program below runs a copy of the tiled kernel without the second `__syncthreads()`, three times, and reports the largest difference from the correct product. Run it, then predict what would happen without the *first* barrier.

In [ ]:
RACE = r"""
__global__ void matmul_racy(const float* A, const float* B, float* C, int n, int k, int m)
{
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y, col = blockIdx.x * TILE + threadIdx.x;
    float acc = 0.0f;
    for (int t = 0; t < (k + TILE - 1) / TILE; ++t) {
        int a_col = t * TILE + threadIdx.x, b_row = t * TILE + threadIdx.y;
        As[threadIdx.y][threadIdx.x] = (row < n && a_col < k) ? A[row * k + a_col] : 0.0f;
        Bs[threadIdx.y][threadIdx.x] = (b_row < k && col < m) ? B[b_row * m + col] : 0.0f;
        __syncthreads();
        for (int r = 0; r < TILE; ++r)
            acc += As[threadIdx.y][r] * Bs[r][threadIdx.x];
        // the bug: no second barrier before the next step overwrites the tiles
    }
    if (row < n && col < m)
        C[row * m + col] = acc;
}

int main()
{
    const int n = 64, k = 64, m = 64;
    static float A[n * k], B[k * m], C[n * m], R[n * m];
    fill(A, n * k, 0.3f);
    fill(B, k * m, 1.7f);
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, sizeof A));
    CUDA_CHECK(cudaMalloc(&d_B, sizeof B));
    CUDA_CHECK(cudaMalloc(&d_C, sizeof C));
    CUDA_CHECK(cudaMemcpy(d_A, A, sizeof A, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, B, sizeof B, cudaMemcpyHostToDevice));
    dim3 block(TILE, TILE), grid(m / TILE, n / TILE);
    matmul_tiled_kernel<<<grid, block>>>(d_A, d_B, d_C, n, k, m);
    CUDA_CHECK(cudaMemcpy(R, d_C, sizeof R, cudaMemcpyDeviceToHost));
    for (int trial = 1; trial <= 3; ++trial) {
        matmul_racy<<<grid, block>>>(d_A, d_B, d_C, n, k, m);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(C, d_C, sizeof C, cudaMemcpyDeviceToHost));
        float worst = 0.0f;
        for (int i = 0; i < n * m; ++i)
            worst = fmaxf(worst, fabsf(C[i] - R[i]));
        std::printf("trial %d: largest error without the second barrier: %g\n", trial, worst);
    }
    return 0;
}
"""
print(run(build(cuda_program([("tiled", matmul_tiled_cu())], RACE), "cuda")))

**What to notice.** Without the second barrier, fast threads load the next tile while slower ones are still reading the current one, so some products use the wrong numbers. The error is usually large. On a GPU it can also change from one run to the next, or even vanish on one run and reappear on another, because it depends on timing. That is a **data race**: two threads access the same memory, at least one writes, and nothing orders them. Without the first barrier, threads would read tiles before they were completely loaded. Tests that pass once do not prove a kernel free of races; reason about every shared-memory access instead.

## 5 · Measure: kernels, transfers and the GPU's speed

GPU work is timed with **CUDA events**, markers that the GPU records in its stream of work as it reaches them: `cudaEventRecord(start)`, the launches, `cudaEventRecord(stop)`, then `cudaEventSynchronize(stop)` and `cudaEventElapsedTime`. Timing with the host's clock would measure only how long the asynchronous launches took to *queue*.

The program measures four things: the naive and the tiled matrix products, as billions of floating-point operations per second (GFLOP/s, counting $2N^3$ for an $N \times N$ product); the bandwidth of copies between host and device; and the bandwidth of a copy kernel within device memory. On a GPU it uses $N$ = 1,024 and `TILE` = 16; with the emulator it uses small sizes and reports CPU timings, which say nothing about a GPU.

In [ ]:
TIMING = r"""
__global__ void copy_kernel(const float* in, float* out, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        out[i] = in[i];
}

static float time_launches(int repeats, cudaEvent_t start, cudaEvent_t stop, bool tiled,
                           const float* d_A, const float* d_B, float* d_C, int N)
{
    dim3 naive_block(16, 16), naive_grid((N + 15) / 16, (N + 15) / 16);
    dim3 tile_block(TILE, TILE), tile_grid((N + TILE - 1) / TILE, (N + TILE - 1) / TILE);
    CUDA_CHECK(cudaEventRecord(start));
    for (int r = 0; r < repeats; ++r) {
        if (tiled)
            matmul_tiled_kernel<<<tile_grid, tile_block>>>(d_A, d_B, d_C, N, N, N);
        else
            matmul_kernel<<<naive_grid, naive_block>>>(d_A, d_B, d_C, N, N, N);
    }
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    float ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
    return ms / repeats;
}

int main()
{
    const int N = SIZE, repeats = REPEATS;
    const size_t bytes = (size_t)N * N * sizeof(float);
    float* A = (float*)std::malloc(bytes);
    float* C1 = (float*)std::malloc(bytes);
    float* C2 = (float*)std::malloc(bytes);
    fill(A, N * N, 0.5f);
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));
    CUDA_CHECK(cudaMemcpy(d_A, A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, A, bytes, cudaMemcpyHostToDevice));
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    time_launches(1, start, stop, false, d_A, d_B, d_C, N);    // warm-up
    float naive_ms = time_launches(repeats, start, stop, false, d_A, d_B, d_C, N);
    CUDA_CHECK(cudaMemcpy(C1, d_C, bytes, cudaMemcpyDeviceToHost));
    time_launches(1, start, stop, true, d_A, d_B, d_C, N);
    float tiled_ms = time_launches(repeats, start, stop, true, d_A, d_B, d_C, N);
    CUDA_CHECK(cudaMemcpy(C2, d_C, bytes, cudaMemcpyDeviceToHost));
    float gap = 0.0f;
    for (int i = 0; i < N * N; ++i)
        gap = fmaxf(gap, fabsf(C1[i] - C2[i]));
    double flops = 2.0 * N * N * N;
    std::printf("%dx%d product, naive: %8.3f ms  %8.1f GFLOP/s\n", N, N, naive_ms, flops / naive_ms / 1e6);
    std::printf("%dx%d product, tiled: %8.3f ms  %8.1f GFLOP/s   (TILE = %d, largest difference %.1e)\n",
                N, N, tiled_ms, flops / tiled_ms / 1e6, TILE, gap);

    const int count = COPY_FLOATS;
    float* big = (float*)std::malloc(count * sizeof(float));
    fill(big, count, 0.1f);
    float *d_in, *d_out;
    CUDA_CHECK(cudaMalloc(&d_in, count * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_out, count * sizeof(float)));
    float ms = 0.0f;
    CUDA_CHECK(cudaEventRecord(start));
    CUDA_CHECK(cudaMemcpy(d_in, big, count * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
    std::printf("host to device copy of %d MB: %8.2f GB/s\n", (int)(count * sizeof(float) >> 20),
                count * sizeof(float) / ms / 1e6);
    copy_kernel<<<(count + 255) / 256, 256>>>(d_in, d_out, count);     // warm-up
    CUDA_CHECK(cudaEventRecord(start));
    copy_kernel<<<(count + 255) / 256, 256>>>(d_in, d_out, count);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
    std::printf("copy kernel within device memory: %8.2f GB/s (bytes read plus bytes written)\n",
                2.0 * count * sizeof(float) / ms / 1e6);
    return 0;
}
"""
on_gpu = CUDA_BACKEND == "gpu"
sizes = {"SIZE": "1024" if on_gpu else "64", "REPEATS": "10" if on_gpu else "2",
         "COPY_FLOATS": str(16 << 20 if on_gpu else 2 << 20)}
timing_source = TIMING
for key, value in sizes.items():
    timing_source = timing_source.replace(key, value)
print("Measured on", GPU_NAME if on_gpu else "the CPU emulator: these are CPU timings, not GPU performance")
print(run(build(cuda_program([("matmul_kernel", matmul_cu()), ("matmul_tiled_kernel", matmul_tiled_cu())],
                             timing_source), "cuda", flags=("-DTILE=16",) if on_gpu else ()), timeout=120))

**What to notice.** On a GPU, a 1,024 × 1,024 product takes on the order of a millisecond: the naive kernel already reaches hundreds of GFLOP/s, and the tiled one is usually faster still, by an amount that depends on the GPU and its caches. Both are far below what NVIDIA's cuBLAS library reaches with deeper tiling, registers and tensor cores. The copy from host to device runs at a few GB/s, while the copy kernel inside device memory moves hundreds of GB/s: crossing the PCIe link costs **tens to hundreds of times more** per byte, which is why training keeps its data on the device. With the emulator every number is a CPU number, and the tiled kernel is slow because each of its blocks starts 64 operating-system threads.

## 6 · Rows and columns: the loss and the bias gradient

A classifier's loss works row by row: each row of logits $Z$ becomes a log-softmax, a loss $\ell_i = \log \sum_j e^{z_{ij}} - z_{i, y_i}$ and a gradient row $(p_i - \text{onehot}(y_i)) / n$. Rows are independent, so the simplest mapping gives each row its own thread, which loops over the row's $c$ classes. With few classes, as here, that is efficient enough; a model with thousands of classes would give each row a whole block.

The bias gradient is a sum **down** each column of the upstream gradient: $\bar b_j = \sum_i \bar Z_{ij}$. Again the simplest mapping gives each column a thread that loops over the rows. Neighbouring threads read neighbouring columns of the same row, so their reads coalesce, and each sum adds its numbers in a fixed order, so the result is the same on every run.

The mean loss then needs the sum of the $n$ row losses: a **reduction**, many numbers into one. The given `sum_kernel` uses the classic pattern with one block: each thread adds up a strided slice, then the block repeatedly halves its partial sums in shared memory, with a barrier between rounds, until thread 0 holds the total. Read it below with the given `add_bias_kernel`, then write the two kernels.

In [ ]:
GIVEN_KERNELS_CU = r"""
__global__ void add_bias_kernel(float* Z, const float* b, int n, int m)   // Z[i][j] += b[j]
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n * m)
        Z[i] += b[i % m];
}

#define REDUCE_THREADS 64
// Adds up n numbers with a single block. Each thread first sums a strided slice;
// then the block halves the number of partial sums in shared memory until one is left.
__global__ void sum_kernel(const float* x, float* total, int n)
{
    __shared__ float partial[REDUCE_THREADS];
    int t = threadIdx.x;
    float s = 0.0f;
    for (int i = t; i < n; i += REDUCE_THREADS)
        s += x[i];
    partial[t] = s;
    __syncthreads();
    for (int stride = REDUCE_THREADS / 2; stride > 0; stride /= 2) {
        if (t < stride)
            partial[t] += partial[t + stride];
        __syncthreads();
    }
    if (t == 0)
        *total = partial[0];
}
"""

### ✍️ Your turn: `softmax_xent_kernel`

Complete `softmax_xent_kernel(Z, y, row_loss, dZ, n, c)`, one thread per row:

- thread `i` (skip it if `i >= n`) takes `const float* z = Z + i * c;`, finds the row's maximum with `fmaxf`, adds up `expf(z[j] - largest)`, and computes `log_total = largest + logf(total)`;
- it writes the row's loss, `row_loss[i] = log_total - z[y[i]]`;
- and its gradient row, `dZ[i * c + j] = (expf(z[j] - log_total) - (j == y[i] ? 1.0f : 0.0f)) / n` for every class `j`.

Use the `float` functions `expf`, `logf` and `fmaxf`, which exist in device code. Returning early is fine here: this kernel has no barrier.

*Type it rather than pasting it: typing is how the formulas sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def softmax_xent_cu_reference():
    return r"""
__global__ void softmax_xent_kernel(const float* Z, const int* y, float* row_loss, float* dZ, int n, int c)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n)
        return;
    const float* z = Z + i * c;
    float largest = z[0];
    for (int j = 1; j < c; ++j)
        largest = fmaxf(largest, z[j]);
    float total = 0.0f;
    for (int j = 0; j < c; ++j)
        total += expf(z[j] - largest);
    float log_total = largest + logf(total);
    row_loss[i] = log_total - z[y[i]];
    for (int j = 0; j < c; ++j)
        dZ[i * c + j] = (expf(z[j] - log_total) - (j == y[i] ? 1.0f : 0.0f)) / n;
}
"""

In [ ]:
def softmax_xent_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void softmax_xent_kernel(const float* Z, const int* y, float* row_loss, float* dZ, int n, int c)
{
    /* ✍️ Your code here: the row index and its test, the maximum, the sum, the loss and the gradient row */
}
"""

In [ ]:
SOFTMAX_MAIN = r"""
static void rows(const char* label, int n, int c, float seed, float scale)
{
    float* Z = (float*)std::malloc(n * c * sizeof(float));
    int* y = (int*)std::malloc(n * sizeof(int));
    float* loss = (float*)std::malloc(n * sizeof(float));
    float* dZ = (float*)std::malloc(n * c * sizeof(float));
    fill(Z, n * c, seed);
    for (int i = 0; i < n * c; ++i)
        Z[i] *= scale;
    for (int i = 0; i < n; ++i)
        y[i] = (i * 7) % c;
    float *d_Z, *d_loss, *d_dZ;
    int* d_y;
    CUDA_CHECK(cudaMalloc(&d_Z, n * c * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_y, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_loss, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_dZ, n * c * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_Z, Z, n * c * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, y, n * sizeof(int), cudaMemcpyHostToDevice));
    softmax_xent_kernel<<<(n + 255) / 256, 256>>>(d_Z, d_y, d_loss, d_dZ, n, c);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(loss, d_loss, n * sizeof(float), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(dZ, d_dZ, n * c * sizeof(float), cudaMemcpyDeviceToHost));
    char name[96];
    std::snprintf(name, sizeof name, "%s: row losses", label);
    show(name, loss, n < 12 ? n : 12);
    std::snprintf(name, sizeof name, "%s: gradient", label);
    show(name, dZ, n * c < 24 ? n * c : 24);
    float total = 0.0f;
    for (int i = 0; i < n; ++i)
        total += loss[i];
    std::snprintf(name, sizeof name, "%s: mean loss", label);
    float mean = total / n;
    show(name, &mean, 1);
    CUDA_CHECK(cudaFree(d_Z));
    CUDA_CHECK(cudaFree(d_y));
    CUDA_CHECK(cudaFree(d_loss));
    CUDA_CHECK(cudaFree(d_dZ));
    std::free(Z); std::free(y); std::free(loss); std::free(dZ);
}

int main()
{
    rows("5 rows of 3", 5, 3, 0.2f, 1.0f);
    rows("logits near +-100", 4, 3, 0.9f, 100.0f);
    rows("300 rows of 4", 300, 4, 1.4f, 2.0f);
    rows("one class", 3, 1, 0.5f, 1.0f);
    return 0;
}
"""


def softmax_program(source):
    return cuda_program([("your softmax_xent_kernel", source)], SOFTMAX_MAIN)


softmax_xent_cu = practice_native("softmax_xent_kernel", softmax_xent_cu, softmax_xent_cu_reference,
                                  softmax_program, language="cuda", atol=1e-6, rtol=1e-5)

**Expected behaviour.** ✅. With logits near ±100 the losses stay finite, because the row maximum was subtracted; without it, `expf(100)` overflows a `float` (whose largest value is about $3.4 \times 10^{38}$, so the threshold is near 88, much lower than a `double`'s 709). With a single class every loss and gradient is 0.

### ✍️ Your turn: `column_sums_kernel`

Complete `column_sums_kernel(M, sums, n, m)`, one thread per column of the $n \times m$ matrix `M`: thread `j` (if `j < m`) adds up `M[i * m + j]` over all rows `i` in a local `float`, then writes it to `sums[j]`. The bias gradient of a layer is exactly `column_sums_kernel` applied to the gradient of its output.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def column_sums_cu_reference():
    return r"""
__global__ void column_sums_kernel(const float* M, float* sums, int n, int m)
{
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j < m) {
        float total = 0.0f;
        for (int i = 0; i < n; ++i)
            total += M[i * m + j];
        sums[j] = total;
    }
}
"""

In [ ]:
def column_sums_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void column_sums_kernel(const float* M, float* sums, int n, int m)
{
    /* ✍️ Your code here: the column index, its test, and the loop over rows */
}
"""

In [ ]:
COLUMNS_MAIN = r"""
static void columns(const char* label, int n, int m, float seed)
{
    const int guard = 16;
    float* M = (float*)std::malloc(n * m * sizeof(float));
    float* sums = (float*)std::malloc((m + guard) * sizeof(float));
    fill(M, n * m, seed);
    for (int j = 0; j < m + guard; ++j)
        sums[j] = -99.0f;
    float *d_M, *d_sums;
    CUDA_CHECK(cudaMalloc(&d_M, n * m * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_sums, (m + guard) * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_M, M, n * m * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_sums, sums, (m + guard) * sizeof(float), cudaMemcpyHostToDevice));
    column_sums_kernel<<<(m + 255) / 256, 256>>>(d_M, d_sums, n, m);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(sums, d_sums, (m + guard) * sizeof(float), cudaMemcpyDeviceToHost));
    int outside = 0;
    for (int j = m; j < m + guard; ++j)
        outside += sums[j] != -99.0f;
    show(label, sums, m);
    std::printf("%s, values written past the end: %d\n", label, outside);
    CUDA_CHECK(cudaFree(d_M));
    CUDA_CHECK(cudaFree(d_sums));
    std::free(M);
    std::free(sums);
}

int main()
{
    columns("1000 rows of 5", 1000, 5, 0.1f);
    columns("3 rows of 300", 3, 300, 2.2f);
    columns("1 row of 7", 1, 7, 0.7f);
    return 0;
}
"""


def columns_program(source):
    return cuda_program([("your column_sums_kernel", source)], COLUMNS_MAIN)


column_sums_cu = practice_native("column_sums_kernel", column_sums_cu, column_sums_cu_reference, columns_program,
                                 language="cuda", atol=1e-5, rtol=1e-5)

**Expected behaviour.** ✅, with nothing written past the end. With one row, the sums are that row itself.

### 🧪 Try it: atomics, order and uninitialised memory

Another way to sum columns gives every *element* its own thread, which adds its value into `sums[j]` with `atomicAdd`, an addition that the hardware performs indivisibly even when many threads target the same address. It parallelises better, but the order of the additions now depends on timing, and floating-point addition is not associative: $(a + b) + c$ can differ from $a + (b + c)$ in the last bits. The program sums 200,000 rows of 4 columns three ways: with your kernel, twice with atomics, and once with atomics into memory that was never cleared.

In [ ]:
ATOMICS = r"""
__global__ void column_sums_atomic(const float* M, float* sums, int n, int m)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n * m)
        atomicAdd(&sums[i % m], M[i]);
}

int main()
{
    const int n = 200000, m = 4;
    float* M = (float*)std::malloc(n * m * sizeof(float));
    fill(M, n * m, 0.4f);
    float *d_M, *d_sums;
    CUDA_CHECK(cudaMalloc(&d_M, n * m * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_M, M, n * m * sizeof(float), cudaMemcpyHostToDevice));
    float fixed[m], first[m], second[m], unset[m];

    CUDA_CHECK(cudaMalloc(&d_sums, m * sizeof(float)));
    column_sums_kernel<<<1, 32>>>(d_M, d_sums, n, m);
    CUDA_CHECK(cudaMemcpy(fixed, d_sums, sizeof fixed, cudaMemcpyDeviceToHost));
    float* runs[2] = {first, second};
    for (float* out : runs) {
        CUDA_CHECK(cudaMemset(d_sums, 0, m * sizeof(float)));     // atomics add to what is there
        column_sums_atomic<<<(n * m + 255) / 256, 256>>>(d_M, d_sums, n, m);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(out, d_sums, m * sizeof(float), cudaMemcpyDeviceToHost));
    }
    float* fresh;
    CUDA_CHECK(cudaMalloc(&fresh, m * sizeof(float)));                // never cleared
    column_sums_atomic<<<(n * m + 255) / 256, 256>>>(d_M, fresh, n, m);
    CUDA_CHECK(cudaMemcpy(unset, fresh, sizeof unset, cudaMemcpyDeviceToHost));
    show("one thread per column, fixed order  ", fixed, m);
    show("atomics, first run                  ", first, m);
    show("atomics, second run                 ", second, m);
    show("atomics into memory never cleared   ", unset, m);
    int same = 0;
    for (int j = 0; j < m; ++j)
        same += first[j] == second[j];
    std::printf("columns where the two atomic runs agree exactly: %d of %d\n", same, m);
    return 0;
}
"""
print(run(build(cuda_program([("column_sums_kernel", column_sums_cu())], ATOMICS), "cuda")))

**What to notice.** All the sums agree to about four significant digits, but on a GPU the atomic runs usually differ from the fixed-order sum, and often from each other, in the last digits. That is harmless for one step, but it makes a training run impossible to reproduce bit for bit, which is why this course's training path uses fixed-order sums. (The CPU emulator runs the threads one after another, so its atomic runs repeat exactly.) The last line shows why device memory must be initialised: atomics add to whatever was there before. The emulator fills new memory with `NaN`, so the bug shows; on a GPU, fresh memory is often zero, so the same bug can go unnoticed until the memory is reused.

## 7 · Put it together: a forward pass on the GPU, checked in double precision

The given host function `forward_loss` runs the whole forward pass of a 2 → 16 → 3 network on the GPU with your kernels: the tiled product, the bias, your ReLU, the second product and bias, your loss kernel, your column sums for the output bias's gradient, and the reduction for the mean loss. It returns the loss, the gradient of the logits and the gradient of the output bias.

The cell compiles your kernels into a shared library and calls `forward_loss` from Python on 300 points of a three-armed spiral with random weights. It then recomputes the same three results in Python with `double` precision, from scratch, and compares. Inside a library, `CUDA_CHECK` must not end the process, because the process is this notebook's kernel; the library's version records the first error, and `forward_loss` returns 1 if there was one.

In [ ]:
LIB_CU = r"""
#include <cmath>
#include <cstdio>

#ifndef TILE
#define TILE 8
#endif

// Inside a library loaded into Python, a failing CUDA call must not end the
// process: CUDA_CHECK records the first error, and each host function reports it.
static cudaError_t first_error = cudaSuccess;
static void record_error(cudaError_t error)
{
    if (error != cudaSuccess && first_error == cudaSuccess)
        first_error = error;
}
#define CUDA_CHECK(call) record_error(call)

extern "C" const char* last_cuda_error() { return cudaGetErrorString(first_error); }
"""

FORWARD_CU = r"""
// One forward pass of the network on the GPU: the loss, the logits' gradient and the
// output bias's gradient. Returns 0, or 1 when a CUDA call failed (see last_cuda_error).
extern "C" int forward_loss(const float* X, const int* y, const float* W1, const float* b1, const float* W2,
                            const float* b2, int n, int d, int h, int c, float* loss, float* dZ2, float* db2)
{
    first_error = cudaSuccess;
    float *dX, *dW1, *db1, *dW2, *db2_in, *Z1, *H, *Z2, *rows, *dZ, *grad_b2, *total;
    int* dy;
    CUDA_CHECK(cudaMalloc(&dX, n * d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dy, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&dW1, d * h * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&db1, h * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dW2, h * c * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&db2_in, c * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&Z1, n * h * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&H, n * h * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&Z2, n * c * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&rows, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dZ, n * c * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&grad_b2, c * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&total, sizeof(float)));
    CUDA_CHECK(cudaMemcpy(dX, X, n * d * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dy, y, n * sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dW1, W1, d * h * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(db1, b1, h * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dW2, W2, h * c * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(db2_in, b2, c * sizeof(float), cudaMemcpyHostToDevice));
    if (first_error != cudaSuccess)
        return 1;

    dim3 tile(TILE, TILE);
    matmul_tiled_kernel<<<dim3((h + TILE - 1) / TILE, (n + TILE - 1) / TILE), tile>>>(dX, dW1, Z1, n, d, h);
    add_bias_kernel<<<(n * h + 255) / 256, 256>>>(Z1, db1, n, h);
    relu_kernel<<<(n * h + 255) / 256, 256>>>(Z1, H, n * h);
    matmul_tiled_kernel<<<dim3((c + TILE - 1) / TILE, (n + TILE - 1) / TILE), tile>>>(H, dW2, Z2, n, h, c);
    add_bias_kernel<<<(n * c + 255) / 256, 256>>>(Z2, db2_in, n, c);
    softmax_xent_kernel<<<(n + 255) / 256, 256>>>(Z2, dy, rows, dZ, n, c);
    column_sums_kernel<<<(c + 255) / 256, 256>>>(dZ, grad_b2, n, c);
    sum_kernel<<<1, REDUCE_THREADS>>>(rows, total, n);
    CUDA_CHECK(cudaGetLastError());

    float summed = 0.0f;
    CUDA_CHECK(cudaMemcpy(&summed, total, sizeof(float), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(dZ2, dZ, n * c * sizeof(float), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(db2, grad_b2, c * sizeof(float), cudaMemcpyDeviceToHost));
    *loss = summed / n;
    float* buffers[] = {dX, dW1, db1, dW2, db2_in, Z1, H, Z2, rows, dZ, grad_b2, total};
    for (float* buffer : buffers)
        CUDA_CHECK(cudaFree(buffer));
    CUDA_CHECK(cudaFree(dy));
    return first_error == cudaSuccess ? 0 : 1;
}
"""

In [ ]:
def spirals(per_class, seed, classes=3, noise=0.2):
    gen = random.Random(seed)
    X, y = array("f"), array("i")
    for c in range(classes):
        for i in range(per_class):
            r = i / per_class
            t = 4.0 * c + 4.0 * r + gen.gauss(0.0, noise)
            X.extend((r * math.sin(t), r * math.cos(t)))
            y.append(c)
    return X, y


def he_weights(fan_in, fan_out, gen):
    return array("f", [gen.gauss(0.0, math.sqrt(2.0 / fan_in)) for _ in range(fan_in * fan_out)])


def forward_reference(X, y, W1, b1, W2, b2, n, d, h, c):
    """The same forward pass in Python, in double precision."""
    total, dZ, db2 = 0.0, [], [0.0] * c
    for i in range(n):
        hidden = [max(0.0, b1[j] + sum(X[i * d + r] * W1[r * h + j] for r in range(d))) for j in range(h)]
        z = [b2[k] + sum(hidden[j] * W2[j * c + k] for j in range(h)) for k in range(c)]
        top = max(z)
        log_total = top + math.log(sum(math.exp(v - top) for v in z))
        total += log_total - z[y[i]]
        row = [(math.exp(v - log_total) - (k == y[i])) / n for k, v in enumerate(z)]
        dZ.extend(row)
        db2 = [a + b for a, b in zip(db2, row)]
    return total / n, dZ, db2


forward_lib = load(LIB_CU + "".join(mark(source, label) for label, source in [
    ("relu", relu_gpu_cu()), ("matmul_tiled_kernel", matmul_tiled_cu()), ("given kernels", GIVEN_KERNELS_CU),
    ("softmax_xent_kernel", softmax_xent_cu()), ("column_sums_kernel", column_sums_cu()),
    ("forward_loss", FORWARD_CU)]), "cuda")
forward_lib.forward_loss.argtypes = [ctypes.c_void_p] * 6 + [ctypes.c_int] * 4 + [ctypes.c_void_p] * 3
forward_lib.forward_loss.restype = ctypes.c_int
forward_lib.last_cuda_error.restype = ctypes.c_char_p

n, d, h, c = 300, 2, 16, 3
X, y = spirals(100, seed=5)
gen = random.Random(6)
W1, b1 = he_weights(d, h, gen), array("f", [0.1] * h)
W2, b2 = he_weights(h, c, gen), array("f", [0.0, 0.2, -0.2])
loss, dZ2, db2 = array("f", [0.0]), array("f", bytes(4 * n * c)), array("f", bytes(4 * c))
status = forward_lib.forward_loss(address(X), address(y), address(W1), address(b1), address(W2), address(b2),
                                  n, d, h, c, address(loss), address(dZ2), address(db2))
assert status == 0, forward_lib.last_cuda_error().decode()

ref_loss, ref_dZ, ref_db2 = forward_reference(X, y, W1, b1, W2, b2, n, d, h, c)
dz_gap = max(abs(a - b) for a, b in zip(dZ2, ref_dZ)) / max(abs(v) for v in ref_dZ)
print(f"loss: GPU {loss[0]:.7f}, double precision {ref_loss:.7f}, relative gap {abs(loss[0] - ref_loss) / ref_loss:.1e}")
print(f"logits' gradient: largest gap {dz_gap:.1e} of its largest value")
print("output bias gradient:", [round(v, 6) for v in db2], "against", [round(v, 6) for v in ref_db2])
assert abs(loss[0] - ref_loss) / ref_loss < 1e-5 and dz_gap < 1e-4, "the GPU forward pass disagrees with the reference"
assert max(abs(a - b) for a, b in zip(db2, ref_db2)) < 1e-5
print("The GPU forward pass agrees with double precision to single-precision accuracy.")

**Expected behaviour.** The loss of the untrained network is about 1.33, a little worse than the $\log 3 \approx 1.10$ of a uniform guess, and the GPU and the double-precision computation agree to about seven significant digits, the precision of a `float`. The output bias's gradient is the column sums of the logits' gradient, so each of its entries is the average of $p - \text{onehot}$ over the rows. Notebook 4 adds the backward pass and the update, and trains the network without leaving the GPU.

## Exercises

Two stretch challenges. Each has a check.

### Exercise 1: `axpy_kernel`, a grid-stride loop

So far every launch had one thread per element. A **grid-stride loop** lets a fixed grid handle any number of elements: each thread starts at its global index and jumps by the total number of threads in the grid, `blockDim.x * gridDim.x`, until it passes `n`. Write `axpy_kernel(a, x, y, n)`, which sets `y[i] = a * x[i] + y[i]` for every `i < n` with such a loop. The check launches it with only 2 blocks of 64 threads on 1,000 elements.

*Type it rather than pasting it: typing is how the pattern sinks in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def axpy_cu_reference():
    return r"""
__global__ void axpy_kernel(float a, const float* x, float* y, int n)
{
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x)
        y[i] = a * x[i] + y[i];
}
"""

In [ ]:
def axpy_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void axpy_kernel(float a, const float* x, float* y, int n)
{
    /* ✍️ Your code here: a grid-stride loop */
}
"""

In [ ]:
AXPY_MAIN = r"""
int main()
{
    const int n = 1000;
    float x[n], y[n];
    fill(x, n, 0.1f);
    fill(y, n, 2.0f);
    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, sizeof x));
    CUDA_CHECK(cudaMalloc(&d_y, sizeof y));
    CUDA_CHECK(cudaMemcpy(d_x, x, sizeof x, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, y, sizeof y, cudaMemcpyHostToDevice));
    axpy_kernel<<<2, 64>>>(3.0f, d_x, d_y, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(y, d_y, sizeof y, cudaMemcpyDeviceToHost));
    show("first 6", y, 6);
    show("last 6", y + n - 6, 6);
    float total = 0.0f;
    for (int i = 0; i < n; ++i)
        total += y[i];
    show("sum", &total, 1);
    return 0;
}
"""


def axpy_program(source):
    return cuda_program([("your axpy_kernel", source)], AXPY_MAIN)


axpy_cu = practice_native("axpy_kernel", axpy_cu, axpy_cu_reference, axpy_program, language="cuda",
                          atol=1e-5, rtol=1e-5)

### Exercise 2: `transpose_kernel`

Write `transpose_kernel(in, out, rows, cols)`, which writes the transpose of the `rows` × `cols` matrix `in` into `out` (`cols` × `rows`): `out[c * rows + r] = in[r * cols + c]`, with one thread per element of `in` on a two-dimensional grid (x for columns, y for rows) and a bounds test. Notice that the reads coalesce but the writes do not, because neighbouring threads write `rows` floats apart. Optimised transposes stage a tile in shared memory so that both sides coalesce.

*Type it rather than pasting it: typing is how the indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def transpose_cu_reference():
    return r"""
__global__ void transpose_kernel(const float* in, float* out, int rows, int cols)
{
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    int r = blockIdx.y * blockDim.y + threadIdx.y;
    if (r < rows && c < cols)
        out[c * rows + r] = in[r * cols + c];
}
"""

In [ ]:
def transpose_cu():
    raise NotImplementedError  # ✍️ delete this line, then complete the kernel below
    return r"""
__global__ void transpose_kernel(const float* in, float* out, int rows, int cols)
{
    /* ✍️ Your code here */
}
"""

In [ ]:
TRANSPOSE_MAIN = r"""
int main()
{
    const int rows = 19, cols = 7;
    float in[rows * cols], out[rows * cols];
    fill(in, rows * cols, 0.8f);
    float *d_in, *d_out;
    CUDA_CHECK(cudaMalloc(&d_in, sizeof in));
    CUDA_CHECK(cudaMalloc(&d_out, sizeof out));
    CUDA_CHECK(cudaMemcpy(d_in, in, sizeof in, cudaMemcpyHostToDevice));
    dim3 block(16, 16), grid((cols + 15) / 16, (rows + 15) / 16);
    transpose_kernel<<<grid, block>>>(d_in, d_out, rows, cols);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(out, d_out, sizeof out, cudaMemcpyDeviceToHost));
    show("19x7 transposed", out, rows * cols);
    return 0;
}
"""


def transpose_program(source):
    return cuda_program([("your transpose_kernel", source)], TRANSPOSE_MAIN)


transpose_cu = practice_native("transpose_kernel", transpose_cu, transpose_cu_reference, transpose_program,
                               language="cuda", atol=0, rtol=0)

## Common errors and debugging

- **`CUDA error: invalid configuration argument`.** The launch asked for more than 1,024 threads per block, or for a grid with a zero dimension, for example 0 blocks for 0 elements. The kernel did not run at all.
- **Results that are all zero or unchanged, with no error.** A launch failed and nobody called `cudaGetLastError()`, or the results were never copied back. Check every launch.
- **`CUDA error: an illegal memory access was encountered`** (on a GPU), reported by a later call. A kernel read or wrote outside an allocation, often through a missing bounds test or a host pointer passed to a kernel. After this error the GPU context is unusable: restart the runtime or kernel, then find the bug with `compute-sanitizer ./program`.
- **`CUDA error: invalid copy direction for memcpy`** or *invalid argument*. The source and destination of `cudaMemcpy` are swapped, or the direction constant does not match the pointers.
- **A tiled kernel that hangs or gives wrong edges.** A thread returned before a `__syncthreads()`, or the tiles are not padded with zeros outside the matrix. Every thread of a block must reach every barrier.
- **Results that change between runs.** A data race, such as a missing barrier, or `atomicAdd` sums in a varying order. The first is a bug; the second is a known source of non-determinism.
- **`nan` from memory you allocated.** `cudaMalloc` does not clear memory; use `cudaMemset` or write every element before reading it. The CPU emulator fills new memory with `NaN` so that this shows.
- **`error: calling a __host__ function from a __global__ function is not allowed`** (from `nvcc`). Device code can call only device functions: use `expf`, `logf` and `fmaxf` rather than `std::exp` or functions from your host code, unless they are marked `__device__`.
- **A Colab cell says the backend is the CPU emulator.** The runtime has no GPU: choose a GPU runtime type and run the notebook again from the top.

## Summary

- A kernel runs once per thread of a grid of blocks. The global index `blockIdx.x * blockDim.x + threadIdx.x` maps threads to data, the block count is rounded up, and a bounds test stops the extra threads.
- The host allocates device memory, copies inputs in, launches, checks `cudaGetLastError()`, copies results out and frees. Launches are asynchronous, and transfers across the PCIe link are slow, so data should stay on the device.
- Neighbouring threads should touch neighbouring addresses, so that their accesses coalesce. One thread per output element, per row or per column is a simple and often adequate mapping.
- Shared memory is a fast, per-block scratchpad. A tiled product loads tiles into it cooperatively, with one barrier before reading and one before overwriting; every thread must reach every barrier.
- CUDA events time GPU work. Data races, missing error checks, out-of-bounds writes, uninitialised memory and atomic ordering are the classic GPU bugs, and several of them stay silent unless you test for them.

Notebook 4 writes the backward pass and the optimiser as kernels and trains the network entirely on the GPU, then checkpoints, evaluates and defends it.